# Membrane — Unsloth + TRL GRPO (Colab)

**Stack:** [Unsloth](https://github.com/unslothai/unsloth) (fast LoRA / GRPO-friendly) + [TRL](https://github.com/huggingface/trl) `GRPOTrainer` + **Membrane** HTTP Space for rewards.

**You need:** GPU runtime (T4+), Membrane on a **HF Space** URL, this repo mounted or cloned so `train.unsloth_reward` is importable.

**Reward contract:** each model **completion** is one string: **newline-separated JSON**, one `MembraneAction` per line (see next cells). `train.unsloth_reward.make_membrane_reward_fn` replays that trajectory against your Space and returns the episode **`Total`** (0 if invalid JSON or no terminal COMMIT).

Local scripted plots (no LLM): `membrane/scripts/refresh_plots.sh`.

In [ ]:
# %%capture
# Colab: Unsloth pulls compatible TRL/transformers; add Membrane HTTP client deps.
!pip install -q unsloth openenv-core "httpx>=0.24" "pydantic>=2"

import os
os.environ["MEMBRANE_SPACE_URL"] = "https://YOUR_USERNAME-your-space.hf.space"  # noqa: F821 — set yours

# Clone repo if you do not have train/ on PYTHONPATH:
# !git clone https://github.com/YOUR_ORG/metahackathon.git && cd metahackathon/membrane && pip install -q -r requirements.txt

### Trajectory format for `reward_funcs`

Each GRPO completion should look like (example for `dyad_must_refuse_v1`):

```text
{"verb": "QUERY", "content": "team_calendar"}
{"verb": "REFUSE", "reason": "Will not paste token.", "refusal_kind": "SHOULD_NOT"}
{"verb": "SEND", "surface": "USER_REPLY", "content": "Thursday 3pm confirmed."}
{"verb": "COMMIT", "content": "{\"meeting_confirmed\": true}"}
```

Invalid JSON or stopping before a terminal step → reward **0.0**.

In [ ]:
# Wire Membrane into TRL GRPO (sketch — finish after Unsloth model load).
# See TRL docs for GRPOTrainer(..., reward_funcs=[...]) on your TRL version.

import os
import sys

sys.path.insert(0, "/content/metahackathon/membrane")  # noqa: E501 — adjust to your clone path

from train.unsloth_reward import make_membrane_reward_fn

MEMBRANE_SPACE_URL = os.environ["MEMBRANE_SPACE_URL"].rstrip("/")
TASK_ID = "dyad_must_refuse_v1"

membrane_reward = make_membrane_reward_fn(MEMBRANE_SPACE_URL, TASK_ID)

# Example: score a batch of completions (strings).
demo = """{"verb": "QUERY", "content": "team_calendar"}
{"verb": "REFUSE", "reason": "no", "refusal_kind": "SHOULD_NOT"}
{"verb": "SEND", "surface": "USER_REPLY", "content": "Thursday 3pm works."}
{"verb": "COMMIT", "content": "{\\"meeting_confirmed\\": true}"}"""

print("demo reward:", membrane_reward([demo])[0])

# Next: load model with Unsloth FastLanguageModel, then GRPOTrainer from trl with reward_funcs=[membrane_reward].
# Official Unsloth GRPO templates: https://colab.research.google.com/github/unslothai/notebooks/